# EyeAI Run 04 — Smart Multi-ROI EfficientNetV2-S

This notebook runs the next HYAMD binary experiment with Smart ROI training and multi-ROI validation.

It includes:
- GitHub clone/pull
- requirements installation
- Hugging Face token setup
- HYAMD preparation
- visual ROI check before training
- EfficientNetV2-S Smart ROI training
- summary display

In [ ]:
from pathlib import Path
import os, subprocess, json, shutil

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"

REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")
OUTPUT_DIR = Path("/kaggle/working/eyeai_binary_ensemble")
REG_CONFIG = "configs/train_efficientnetv2_binary_smart_roi_fixed_unfreeze.yaml"

print("REPO_DIR:", REPO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("REG_CONFIG:", REG_CONFIG)

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print("Repository exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print("Cloning repository...")
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Current directory:", Path.cwd())
for p in sorted(REPO_DIR.iterdir()):
    print("-", p.name)

In [ ]:
os.chdir(REPO_DIR)

required_files = [
    "requirements.txt",
    "pyproject.toml",
    REG_CONFIG,
    "scripts/prepare_hyamd.py",
    "scripts/train_binary.py",
    "src/eyeai/data/transforms.py",
    "src/eyeai/training/train_binary.py",
]

missing = []
for file_path in required_files:
    exists = (REPO_DIR / file_path).exists()
    print(f"{file_path}: {exists}")
    if not exists:
        missing.append(file_path)

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")
print("Repository structure is OK.")

In [ ]:
os.chdir(REPO_DIR)
print("Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Installing EyeAI package in editable mode...")
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("EyeAI package installed correctly.")

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN secret is missing.")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["HF_HUB_TOKEN"] = hf_token

print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))

try:
    from huggingface_hub import login, HfApi
    login(token=hf_token, add_to_git_credential=False)
    who = HfApi().whoami(token=hf_token)
    print("Hugging Face authentication OK.")
    print("HF user:", who.get("name", "unknown"))
except Exception as e:
    print("Hugging Face login check failed:")
    print(repr(e))

In [ ]:
# Clean incomplete Hugging Face downloads and locks
from pathlib import Path

cache_dirs = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path("/kaggle/working/.cache/huggingface/hub"),
]

removed = 0
for cache_dir in cache_dirs:
    if not cache_dir.exists():
        continue
    for pattern in ["*.incomplete", "*.lock"]:
        for p in cache_dir.rglob(pattern):
            try:
                p.unlink()
                removed += 1
                print("Removed:", p)
            except Exception as e:
                print("Could not remove:", p, repr(e))
print("Removed incomplete/lock files:", removed)

In [ ]:
# Optional: pre-download only the EfficientNetV2-S weights file.
# If HF is slow in this session, skip this cell and let timm download during model creation.
import os
from huggingface_hub import hf_hub_download

repo_id = "timm/tf_efficientnetv2_s.in21k_ft_in1k"
print("Downloading only model.safetensors from:", repo_id)

weights_path = hf_hub_download(
    repo_id=repo_id,
    filename="model.safetensors",
    token=os.environ.get("HF_TOKEN"),
    resume_download=True,
    force_download=False,
)
print("Downloaded weights:")
print(weights_path)

In [ ]:
# Check Kaggle input paths and config
from pathlib import Path
import yaml

os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Kaggle input folders:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print("-", p)

config_path = REPO_DIR / REG_CONFIG
print("Config path:", config_path)
print("Config exists:", config_path.exists())

with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Top-level config keys:", list(cfg.keys()))
print("Data config:", cfg.get("data", {}))

hyamd_path = Path(cfg.get("data", {}).get("input_dir", ""))
print("HYAMD path:", hyamd_path)
print("Exists:", hyamd_path.exists())

if not hyamd_path.exists():
    raise FileNotFoundError(f"HYAMD input path does not exist: {hyamd_path}")

In [ ]:
# Prepare HYAMD data and binary splits
os.chdir(REPO_DIR)
!python -u scripts/prepare_hyamd.py --config configs/train_efficientnetv2_binary_smart_roi_fixed_unfreeze.yaml

In [ ]:
# Inspect generated splits
import pandas as pd

split_dir = OUTPUT_DIR / "HYAMD_raw" / "splits"
print("Split directory:", split_dir)
print("Exists:", split_dir.exists())

for name in ["train.csv", "val.csv", "test.csv"]:
    p = split_dir / name
    print("
" + "=" * 80)
    print(name, "exists:", p.exists())
    if p.exists():
        df = pd.read_csv(p)
        print("shape:", df.shape)
        if "binary_label" in df.columns:
            print(df["binary_label"].value_counts().sort_index())
        if "label" in df.columns:
            print(df["label"].value_counts().sort_index())

In [ ]:
# Visual check: original fundus crop vs Smart ROI views after no-black ROI adjustment
from pathlib import Path
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import yaml

# Make sure the local package is importable inside Kaggle notebooks.
os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from eyeai.data.transforms import ROISpec, crop_by_roi_spec, _black_fraction

split_dir = OUTPUT_DIR / "HYAMD_raw" / "splits"
train_csv = split_dir / "train.csv"
df = pd.read_csv(train_csv)
image_col = "proc_image_path" if "proc_image_path" in df.columns else "image_path"

with open(REPO_DIR / REG_CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

roi_specs = cfg["data"].get("roi_specs", [])
black_fill_mode = cfg["data"].get("black_fill_mode", "none")
black_threshold = int(cfg["data"].get("black_threshold", 8))
avoid_black_roi = bool(cfg["data"].get("avoid_black_roi", True))
max_black_fraction = float(cfg["data"].get("max_black_fraction", 0.015))
min_roi_scale = float(cfg["data"].get("min_roi_scale", 0.45))

print("black_fill_mode:", black_fill_mode)
print("black_threshold:", black_threshold)
print("avoid_black_roi:", avoid_black_roi)
print("max_black_fraction:", max_black_fraction)
print("min_roi_scale:", min_roi_scale)
print("ROI specs:", [r.get("name") for r in roi_specs])

sample_df = df.sample(n=2, random_state=42).reset_index(drop=True)

display_specs = [{"name": "original", "cx": 0.5, "cy": 0.5, "scale": 1.0}] + roi_specs
cols = 4
rows_per_image = (len(display_specs) + cols - 1) // cols
rows = len(sample_df) * rows_per_image
plt.figure(figsize=(4 * cols, 4 * rows))

plot_idx = 1
for _, row in sample_df.iterrows():
    img_path = Path(row[image_col])
    img = Image.open(img_path).convert("RGB")
    image_name = row.get("image_name", img_path.name)
    label = row.get("label", "?")
    binary_label = row.get("binary_label", "?")

    for spec_dict in display_specs:
        spec = ROISpec(
            name=str(spec_dict.get("name", "roi")),
            cx=float(spec_dict.get("cx", 0.5)),
            cy=float(spec_dict.get("cy", 0.5)),
            scale=float(spec_dict.get("scale", 1.0)),
        )

        if spec.name == "original":
            view = img
            bf = _black_fraction(view, threshold=black_threshold)
            title = f"Original\n{image_name}\nlabel={label}, binary={binary_label}\nblack={bf:.3f}"
        else:
            view = crop_by_roi_spec(
                img,
                spec,
                black_fill_mode=black_fill_mode,
                black_threshold=black_threshold,
                avoid_black_roi=avoid_black_roi,
                max_black_fraction=max_black_fraction,
                min_roi_scale=min_roi_scale,
            )
            bf = _black_fraction(view, threshold=black_threshold)
            title = f"{spec.name}\nrequested scale={spec.scale}\nblack={bf:.3f}"

        plt.subplot(rows, cols, plot_idx)
        plt.imshow(view)
        plt.title(title, fontsize=8)
        plt.axis("off")
        plot_idx += 1

    while (plot_idx - 1) % cols != 0:
        plt.subplot(rows, cols, plot_idx)
        plt.axis("off")
        plot_idx += 1

plt.tight_layout()
plt.show()


In [ ]:
# Train EfficientNetV2-S Smart ROI run
os.chdir(REPO_DIR)
!python -u scripts/train_binary.py --config configs/train_efficientnetv2_binary_smart_roi_fixed_unfreeze.yaml

In [ ]:
# Show generated output files
root = OUTPUT_DIR
print("Output root:", root)
print("Exists:", root.exists())
important_parts = ["checkpoints", "logs", "predictions"]
for p in sorted(root.rglob("*")):
    if p.is_file() and any(part in str(p) for part in important_parts):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{p} | {size_mb:.2f} MB")

In [ ]:
# Read Smart ROI training summaries
import json
import pandas as pd

logs_dir = OUTPUT_DIR / "logs"
summary_files = sorted(logs_dir.glob("*smart_roi*summary*.json"))
history_files = sorted(logs_dir.glob("*smart_roi*history*.csv"))

print("Summary files:")
for p in summary_files:
    print("-", p)

print("History files:")
for p in history_files:
    print("-", p)

for p in summary_files:
    print("
" + "=" * 90)
    print("SUMMARY:", p.name)
    with open(p, "r", encoding="utf-8") as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2, ensure_ascii=False))

if history_files:
    latest_history = history_files[-1]
    print("
" + "=" * 90)
    print("Latest history:", latest_history)
    hist = pd.read_csv(latest_history)
    display(hist.tail(10))